# Demucs 구조 뜯어보기 (HTDemucs): 수업의 CNN · 인코더–디코더 · Attention · Transformer가 실제 모델에서 어디에 있나

*2026-11-06 작성, 이슈 #35의 확장. 강사님 면담(원리 이해가 가장 중요)에 따라 "모델 구조 이해"를 첫 탐구로 정했습니다.*

이 노트북은 **내가 이미 앱에서 쓰는 보컬 분리 모델**(`separate.py`의 `get_model("htdemucs")`)을 열어서, `apply_model(...)` 한 줄 뒤에서 일어나는 일을 **직접 확인**하는 탐구 노트입니다. Keras로는 몇 줄이지만 실제로 어떤 원리로 동작하는지 이해하는 것이 목표입니다.

## 한눈에 보기

| 단계 | 질문 | 어떻게 확인하나 | 얻는 것 |
|---|---|---|---|
| 1 | 앱의 코드 한 줄이 라이브러리의 어느 클래스로 이어지나? | `inspect`로 소스 위치와 호출 경로 추적 | 호출 경로 |
| 2 | 모델은 어떤 덩어리로 되어 있고 파라미터는 얼마나 되나? | `named_children()`로 모듈 트리와 파라미터 수 | 구조 표 |
| 3 | 입력은 어떻게 만드나? | 직접 만든 합성 믹스(정답을 알고 있음) | 실험 입력 |
| 4 | 데이터의 모양은 층마다 어떻게 변하나? | `forward hook`으로 층별 입력·출력 모양 기록 | **텐서 모양 표** |
| 5 | Attention은 어디에서 무엇과 무엇을 연결하나? | 어텐션 모듈에 hook, 질의(query)·키(key) 길이 | self/cross 표 |
| 6 | 분리가 실제로 되었나? 고정 필터와 무엇이 다른가? | 정답이 있으므로 SDR 계산, 대역통과 필터와 비교 | 숫자 |
| 7 | 내가 이해한 것 / 아직 모르는 것 | 직접 채우기 | **강사님께 할 질문** |

**표기**: **[확인]** 이 노트북 코드로 직접 확인한 것, **[일반 지식]** 논문·문서에서 일반적으로 알려진 것(확인 필요), **[추정]** 내 추측, **[미확인]** 아직 못 한 것.

**데이터 규칙**: 상업곡의 음원·가사는 쓰지 않습니다. 입력은 **내가 직접 쓴 예문의 TTS 소리 + 직접 만든 합성 반주**뿐입니다(저작권 문제 없음). 그래서 이 노트북의 출력은 커밋해도 안전하지만, 습관대로 출력을 지운 정리본을 커밋합니다.

**걸리는 시간**: 환경 확인~구조 표는 몇 초, **4단계(모델 한 번 실행)는 CPU에서 약 1분**입니다(GPU 없음). 위에서 아래로 실행하세요.

## 0. 먼저 가설: 수업 개념은 Demucs의 어디에 있을까? (아래 실험으로 하나씩 확인)

| 수업 | 개념 | Demucs에서 찾을 곳 (가설) |
|---|---|---|
| 02·03강 | FFT, STFT, 스펙트로그램 | 스펙트로그램 가지의 입력(`nfft=4096`, `hop=1024`) |
| 006 CNN | 합성곱으로 국소 패턴 추출, 채널↑·크기↓ | 인코더 4층(`Conv`), 채널이 늘고 길이가 줄어듦 |
| 010 Seq2Seq | 인코더로 압축 → 디코더로 복원 | `encoder`/`decoder`, 같은 단계끼리 **건너뛰기 연결**(U-Net) |
| 011 Attention | $\mathrm{softmax}(QK^\top/\sqrt d)V$ | `crosstransformer` 안의 `MultiheadAttention` |
| 012 Transformer | 셀프 어텐션 + 피드포워드 층 쌓기 | `MyTransformerEncoderLayer`, `CrossTransformerEncoderLayer` |
| 009 RNN/LSTM | 순서대로 상태 전달 | 이전 세대 Demucs(BLSTM). **현재 모델에 LSTM이 있나?** → 3단계에서 셉니다 |

In [ ]:
# ── 환경 확인 ──────────────────────────────────────────────────────────────
# 하는 일: 프로젝트 루트를 찾아 작업 폴더로 하고(노트북이 experiments/ 아래에 있어도 루트의 모듈을 import하려고),
#          demucs·torch 버전과 장치(CPU)를 출력한다. [수업에 없음] demucs, torch(수업은 Keras/TensorFlow)
import os, sys, time, inspect
from pathlib import Path
here = Path.cwd().resolve()
for p in [here, *here.parents]:
    if (p / "practice.py").exists():
        os.chdir(p); sys.path.insert(0, str(p)); break
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from IPython.display import display      # 표를 예쁘게 보여 주는 노트북 함수
import torch, demucs
import theme, separate, visualisation
theme.apply_style()
print("프로젝트 루트:", Path.cwd().name, "| demucs", demucs.__version__, "| torch", torch.__version__, "| 장치: CPU(GPU 없음)")

## 1. 앱의 코드 한 줄 → 라이브러리의 어느 클래스?

`separate.py`가 하는 일은 사실 세 줄입니다: **모델을 가져오고(`get_model`) → 추론 모드로 바꾸고(`eval`) → 적용한다(`apply_model`)**. 이 셀에서 그 호출 경로와 실제 소스 파일 위치를 찾습니다.

In [ ]:
# ── 1단계: 호출 경로 따라가기 ────────────────────────────────────────────────
# 하는 일: separate.py의 두 함수 소스를 보여 주고, get_model("htdemucs")가 돌려주는 객체의 종류를 확인한다.
print(inspect.getsource(separate._load_model))
from demucs.pretrained import get_model
bag = get_model("htdemucs"); bag.eval()          # eval(): 드롭아웃 등을 끄는 추론 모드 (수업 6강 model.predict와 같은 상태)
print("get_model이 돌려준 것:", type(bag).__name__, "| 안에 든 모델 수:", len(bag.models), "| 가중치:", bag.weights)
model = bag.models[0]
print("실제 모델 클래스:", type(model).__name__)
print("분리하는 소리:", model.sources, "| 표본화 주파수:", model.samplerate, "Hz | 학습 구간 길이:", float(model.segment), "초")

In [ ]:
# ── 소스 위치 ──────────────────────────────────────────────────────────────
# 하는 일: 우리가 읽어야 할 클래스가 라이브러리의 어느 파일 몇 번째 줄에 있는지 알려 준다(나중에 직접 열어 읽으려고).
from demucs import htdemucs, transformer
targets = {
    "HTDemucs (전체 모델, forward)": htdemucs.HTDemucs,
    "CrossTransformerEncoder (가운데 변환기 묶음)": transformer.CrossTransformerEncoder,
    "MyTransformerEncoderLayer (셀프 어텐션 층)": transformer.MyTransformerEncoderLayer,
    "CrossTransformerEncoderLayer (교차 어텐션 층)": transformer.CrossTransformerEncoderLayer,
}
for name, obj in targets.items():
    path = inspect.getsourcefile(obj); line = inspect.getsourcelines(obj)[1]
    print(f"{name:48s} {Path(path).parent.name}/{Path(path).name}:{line}")

## 2. 구조 한눈에: 모델은 어떤 덩어리로 되어 있나

`named_children()`는 모델 바로 아래 덩어리를 보여 줍니다. 파라미터 수는 "그 덩어리가 학습으로 정해진 숫자를 몇 개 갖고 있나"입니다.

**[확인: 11-06 실행]** 총 약 **4,198만 개**, 소리 4개(drums·bass·other·vocals), **`nn.LSTM`은 0개**(→ 현재 모델에 LSTM 없음, 소스에는 이전 세대의 `BLSTM` 클래스가 남아 있음).

**[확인: 11-06 실행] 파라미터의 약 75%(3,155만 개)가 가운데 `crosstransformer`에 있습니다.** 인코더·디코더(CNN)는 합쳐서 약 22%입니다. → 이 모델에서 학습된 숫자의 대부분은 **Attention(Transformer) 쪽**에 있다는 뜻이고, 수업의 CNN만으로는 이 모델의 무게중심을 설명하지 못합니다.

In [ ]:
# ── 2단계: 모듈 트리와 파라미터 수 ───────────────────────────────────────────
rows = []
for name, child in model.named_children():
    n = sum(p.numel() for p in child.parameters())
    rows.append({"덩어리": name, "클래스": type(child).__name__, "층 수": len(child) if hasattr(child, "__len__") else "", "파라미터(만 개)": round(n / 1e4, 1)})
df = pd.DataFrame(rows)
total = sum(p.numel() for p in model.parameters())
df["비율(%)"] = (df["파라미터(만 개)"] * 1e4 / total * 100).round(1)
display(df)
print(f"총 파라미터 {total/1e6:.1f}M")
count = lambda cls: sum(isinstance(m, cls) for m in model.modules())
print("MultiheadAttention:", count(torch.nn.MultiheadAttention), "| Conv1d:", count(torch.nn.Conv1d), "| Conv2d:", count(torch.nn.Conv2d), "| LSTM:", count(torch.nn.LSTM))
print("스펙트로그램 설정: nfft =", model.nfft, ", hop =", model.hop_length, ", cac =", model.cac, "(cac=True: 복소수를 실수 2채널로 풀어 쌓음)")

### 구조 개념도 (소스를 읽은 뒤의 내 정리 — 아래 4단계의 실제 모양과 맞춰 봅니다)

```text
             섞인 소리 (2채널, 44.1 kHz)
                   │
        ┌──────────┴───────────┐
   [시간(파형) 가지]       [스펙트로그램 가지]
   tencoder 4층 (Conv1d)   STFT(nfft 4096) → 실수 4채널 (cac)
        │                  encoder 4층 (Conv2d)
        │                       │
        └──── crosstransformer (5층) ────┘      ← 두 가지의 정보를 Attention으로 주고받음
              self-attn → cross-attn → self → cross → self
        ┌──────────┴───────────┐
   tdecoder 4층 (ConvTranspose1d)   decoder 4층 (ConvTranspose2d)
        │      ↑ 같은 단계의 인코더 출력을 더함(건너뛰기 연결)      │
        └──────── 두 가지 결과를 더함 ────────┘
                   │
         drums / bass / other / vocals
```

- **왜 두 가지(파형 + 스펙트로그램)로 보나?** 파형은 시간 모양(타악기의 순간적인 타격 등)을, 스펙트로그램은 주파수 구성(배음 줄무늬 등)을 잘 보여 주고, 서로 정보가 다르기 때문입니다 [일반 지식, 논문의 동기 설명].

## 3. 실험 입력 만들기: 정답을 아는 믹스

상업곡은 "정답 보컬"이 없어서 분리 품질을 숫자로 못 잽니다. 그래서 **내가 직접 쓴 예문의 TTS 소리(= 정답 "목소리")** 에 **직접 만든 합성 반주(저음 + 타악기 소리)** 를 더해 믹스를 만듭니다. 그러면 분리 결과를 정답과 비교할 수 있습니다(SDR, 6단계).

**주의**: Demucs는 *노래* 보컬로 학습한 모델이라 말소리·합성 반주에서는 실제 곡과 다르게 동작할 수 있습니다. 이 노트북의 숫자는 "구조 관찰과 평가 방법 연습"용이고, 실제 곡의 성능으로 일반화하지 않습니다.

In [ ]:
# ── 3단계: 합성 믹스 만들기 ──────────────────────────────────────────────────
# 하는 일: data/tts_cache/s01.wav(직접 쓴 예문 1번의 TTS)를 44.1 kHz 스테레오로 읽어 3초를 목소리(정답)로 쓰고,
#          55 Hz 저음 + 0.5초마다 울리는 잡음 타격을 반주로 더해 7.8초 믹스를 만든다(7.8초 = 모델의 학습 구간 길이).
SR = separate.MODEL_SR                                  # 44100
N = int(float(model.segment) * SR)                       # 343980 표본
voice = np.zeros((2, N), np.float32)
speech = separate.load_clip("data/tts_cache/s01.wav", 0.0, 3.0)    # (2, 표본): 앱의 구간 읽기 함수 재사용
voice[:, :speech.shape[1]] = speech[:, :N]
t = np.arange(N) / SR
rng = np.random.default_rng(0)                           # 시드 고정: 매번 같은 반주
bass = 0.2 * np.sin(2 * np.pi * 55 * t)
drums = np.zeros(N)
for k in range(0, N, SR // 2):
    L = min(int(0.05 * SR), N - k)
    drums[k:k + L] += rng.standard_normal(L) * np.exp(-np.arange(L) / (0.01 * SR)) * 0.4
accomp = np.stack([bass + drums] * 2).astype(np.float32)
mix = voice + accomp
fig, ax = plt.subplots(3, 1, figsize=(9, 4.5), sharex=True)
for a, x, name in zip(ax, [voice[0], accomp[0], mix[0]], ["목소리(정답)", "합성 반주", "섞은 소리(모델 입력)"]):
    a.plot(t, x, lw=0.4, color=theme.WAVEFORM); a.set_ylabel(name, rotation=0, ha="right", va="center")
ax[-1].set_xlabel("시간(초)"); plt.tight_layout(); plt.show()
print("믹스 모양:", mix.shape, "| 길이", round(N / SR, 2), "초")

## 4. 데이터의 모양은 층마다 어떻게 변하나 (forward hook)

**forward hook**은 "층이 계산을 끝낼 때마다 나에게 알려 줘"라고 걸어 두는 함수입니다. 모델을 고치지 않고 층의 입력·출력 모양을 기록할 수 있습니다. 여기서는 인코더·디코더·가운데 변환기·어텐션 모듈에 hook을 겁니다. **이 셀이 모델을 실제로 한 번 실행합니다(CPU 약 1분).** 이 한 번의 실행 결과가 6단계의 분리 결과로도 쓰입니다.

[수업에 없음] `register_forward_hook`: PyTorch의 기능으로, Keras의 중간 층 출력을 보려고 `Model(inputs, layer.output)`을 만드는 것과 같은 목적입니다.

In [ ]:
# ── 4단계: hook을 걸고 모델을 한 번 실행 ──────────────────────────────────────
# 하는 일: (1) 인코더/디코더/변환기 층의 입력·출력 모양, (2) 어텐션 모듈의 질의(query)·키(key) 모양을 기록한다.
#          (3) 같은 실행의 결과를 stems(분리된 4개 소리)로 저장한다. 입력 정규화는 separate.separate_vocals와 같은 방식.
def shape_of(o):
    if isinstance(o, torch.Tensor): return tuple(o.shape)
    if isinstance(o, (tuple, list)): return [shape_of(i) for i in o]
    return None
layer_rec, attn_rec, hooks = [], [], []
layer_names = [f"{p}.{i}" for p in ("tencoder", "encoder", "decoder", "tdecoder") for i in range(4)] + ["crosstransformer"]
for name, mod in model.named_modules():
    if name in layer_names:
        hooks.append(mod.register_forward_hook(lambda m, a, o, n=name: layer_rec.append((n, [shape_of(x) for x in a if x is not None and not isinstance(x, int)], shape_of(o)))))
    if isinstance(mod, torch.nn.MultiheadAttention):
        def h(m, args, kwargs, out, n=name):
            q = args[0] if args else kwargs["query"]; k = args[1] if len(args) > 1 else kwargs["key"]
            attn_rec.append((n, tuple(q.shape), tuple(k.shape)))
        hooks.append(mod.register_forward_hook(h, with_kwargs=True))

x = torch.from_numpy(mix)[None]                                  # (1, 2, 표본) = 배치 1개
ref = x.mean(1); mean, std = ref.mean(), ref.std()
t0 = time.time()
with torch.no_grad():                                            # 기울기 계산을 끄면 메모리·시간이 줄어든다
    stems = model((x - mean) / std) * std + mean                 # (1, 4, 2, 표본): 4개 소리
for h_ in hooks: h_.remove()                                     # hook 정리(다음 실행에 영향 없게)
print(f"모델 한 번 실행: {time.time() - t0:.0f}초 | 출력 모양 {tuple(stems.shape)} = (배치, 소리 4개, 채널 2, 표본)")

In [ ]:
# ── 층별 모양 표 ─────────────────────────────────────────────────────────────
# 하는 일: 기록한 모양을 표로 보여 준다. 파형 가지는 (배치, 채널, 시간), 스펙트로그램 가지는 (배치, 채널, 주파수, 시간) 모양이다.
order = {n: i for i, n in enumerate(layer_names)}
tab = pd.DataFrame([{"층": n, "입력 모양": s_in[0] if len(s_in) == 1 else s_in, "출력 모양": s_out} for n, s_in, s_out in layer_rec])
display(tab)

### 모양이 알려 주는 것 (아래 표의 숫자와 직접 대조하세요)

**[확인: 11-06 실행 값, 7.8초 믹스]**
- **시간(파형) 가지** `tencoder`: 길이 343,980 → 85,995 → 21,499 → 5,375 → 1,344 (한 층마다 **약 1/4**), 채널 2 → 48 → 96 → 192 → 384 (**약 2배**). → 수업 CNN의 "합성곱+스트라이드로 크기↓ 채널↑"과 같은 모양입니다.
- **스펙트로그램 가지** `encoder`: 입력 (4, 2048, 336)에서 주파수 칸이 2048 → 512 → 128 → 32 → 8(**1/4씩**), 시간 칸(336)은 유지, 채널 4 → 48 → 96 → 192 → 384.
  - 입력 채널 **4** = 스테레오 2채널 × (실수부, 허수부) — `cac=True` 때문입니다 [확인: 설정값, 해석은 소스 기준].
  - 주파수 칸 **2048** = `nfft 4096`의 절반입니다(수업 FFT에서 한쪽 반만 쓰는 것과 같은 이유 [일반 지식]). 시간 칸 **336** ≈ 343,980 ÷ `hop 1024`.
- **디코더**는 인코더의 반대로 길이·주파수 칸을 다시 4배씩 키우고 채널을 줄입니다. 마지막 `tdecoder.3`의 출력 (8, 343,980) = **소리 4개 × 채널 2**가 원래 길이로 복원된 것입니다.
- **가운데 `crosstransformer`**: 두 가지가 같은 크기의 채널(512)로 맞춰져 들어가고, 스펙트로그램 가지는 (512, 8, 336), 시간 가지는 (512, 1344)입니다. → **변환기가 보는 "토큰(조각)의 수"** 가 가지마다 다릅니다(아래 5단계).

## 5. Attention은 어디에서 무엇과 무엇을 연결하나

`crosstransformer` 안에는 어텐션이 **10개**(스펙트로그램 쪽 5 + 시간 쪽 5) 있습니다. 층 이름의 `self_attn` / `cross_attn`을 보면 어떤 종류인지 알 수 있습니다.
- **셀프 어텐션**: 질의(Q), 키(K), 값(V)가 **같은 가지**에서 나옴 → 같은 소리 안에서 멀리 떨어진 부분끼리의 관계.
- **교차 어텐션**: 질의는 한 가지에서, 키·값은 **다른 가지**에서 → 파형 가지가 스펙트로그램 가지를 "참조"하는 식으로 정보를 교환 [일반 지식, 논문 설명].

In [ ]:
# ── 5단계: 어텐션 모듈의 질의·키 길이 ────────────────────────────────────────
# 하는 일: 4단계에서 hook으로 기록한 값을 표로 보여 준다. 질의 길이 ≠ 키 길이면 교차 어텐션이다.
at = pd.DataFrame(attn_rec, columns=["모듈", "질의(Q) 모양", "키(K) 모양"])
at["종류"] = at["모듈"].str.contains("cross_attn").map({True: "교차(cross)", False: "셀프(self)"})
at["가지"] = at["모듈"].str.contains("layers_t").map({True: "시간(파형)", False: "스펙트로그램"})
display(at[["가지", "모듈", "종류", "질의(Q) 모양", "키(K) 모양"]])

**읽는 법**: 모양은 보통 `(배치, 토큰 수, 차원)`입니다. 스펙트로그램 가지의 토큰 수는 **주파수 칸 × 시간 칸 = 8 × 336 = 2,688**, 시간 가지는 **1,344**입니다 [확인: 위 표의 질의 모양과 대조]. 교차 어텐션에서는 질의 쪽 토큰 수와 키 쪽 토큰 수가 다릅니다(= 서로 다른 표현의 정보를 섞는다는 증거).

### 수업의 Attention 식을 눈으로: 토큰 4개짜리 작은 예

위 어텐션 모듈 안에서 일어나는 계산의 핵심은 수업 011강의 이 식입니다. **토큰 4개**로 직접 계산해 봅니다.

$$\mathrm{Attention}(Q,K,V)=\mathrm{softmax}\!\left(\frac{QK^{\top}}{\sqrt{d}}\right)V$$

In [ ]:
# ── 작은 어텐션 계산 (numpy) ─────────────────────────────────────────────────
# 하는 일: 토큰 4개, 차원 8의 Q·K·V로 점수 → softmax → 가중합을 직접 계산한다. 실제 모델은 이것을 토큰 수천 개, 헤드 여러 개로 한다.
rng = np.random.default_rng(1)
Q, K, V = (rng.standard_normal((4, 8)) for _ in range(3))
score = Q @ K.T / np.sqrt(Q.shape[1])                    # (4, 4): 질의 i가 키 j를 얼마나 보나
w = np.exp(score - score.max(1, keepdims=True)); w /= w.sum(1, keepdims=True)     # softmax: 각 줄의 합이 1
out = w @ V                                               # (4, 8): 값(V)의 가중합
print("가중치(각 줄의 합 = 1):\n", w.round(2), "\n줄 합:", w.sum(1).round(2))
print("출력 모양:", out.shape, "| 질의 4개 각각이 '키 4개'를 섞어 새 벡터를 만든다")

**연산량 감각** [확인: 값은 계산 가능]: 스펙트로그램 가지에서 토큰이 2,688개이면 점수 표는 2,688 × 2,688 ≈ **720만 칸**입니다. 토큰 수의 제곱으로 늘기 때문에 긴 소리를 한 번에 못 넣고 **7.8초 조각으로 나눠 처리**하는 이유 중 하나일 것입니다 [추정]. 앱의 `separate_vocals`가 `split=True, overlap=0.25`로 조각을 겹쳐 처리하는 것도 같은 이유와 이어집니다.

## 6. 분리가 실제로 되었나? 고정 필터와 무엇이 다른가 (#35의 ②·③ 미리 보기)

정답(목소리)을 알고 있으므로 **SDR**(신호 대 왜곡비, 클수록 정답에 가까움)로 비교합니다. $\mathrm{SDR}=10\log_{10}\frac{\sum \text{정답}^2}{\sum(\text{정답}-\text{추정})^2}$ [일반 지식: 음원 분리 평가의 기본형].
비교 대상: ① Demucs의 `vocals` ② 섞은 소리 그대로 ③ **고정 대역통과 필터**(200~4,000 Hz만 통과 — "목소리와 악기는 주파수 음역대가 달라서 분리된다"는 가설의 가장 단순한 구현).

In [ ]:
# ── 6단계-1: SDR 비교 ────────────────────────────────────────────────────────
# 하는 일: 정답(voice)과 각 추정의 SDR을 계산한다. scipy.signal.butter: 버터워스 대역통과 필터(4차), sosfiltfilt: 앞뒤로 두 번 걸어 위상이 밀리지 않게.
# [수업에 없음] SDR, 버터워스 필터(수업은 FFT로 주파수를 보는 데까지)
from scipy.signal import butter, sosfiltfilt
def sdr(ref, est):
    return 10 * np.log10((ref ** 2).sum() / (((ref - est) ** 2).sum() + 1e-12))
voc = stems[0, model.sources.index("vocals")].numpy()
sos = butter(4, [200, 4000], btype="bandpass", fs=SR, output="sos")
bp = sosfiltfilt(sos, mix, axis=1).astype(np.float32)
res = pd.DataFrame({"방법": ["섞은 소리 그대로", "고정 대역통과 200~4000 Hz", "Demucs vocals"],
                    "SDR(dB)": [sdr(voice, mix), sdr(voice, bp), sdr(voice, voc)]}).round(2)
display(res)
print("SDR이 클수록 정답 목소리에 가깝다. Demucs가 필터보다 크면 '주파수 음역대만으로는 분리가 안 된다'는 쪽의 증거(이 합성 입력에서).")
for s_name in model.sources:
    e = stems[0, model.sources.index(s_name)].numpy()
    print(f"{s_name:7s} RMS = {np.sqrt((e ** 2).mean()):.4f}")

In [ ]:
# ── 6단계-2: 대역별 에너지 비율 (#35 ②의 작은 버전) ──────────────────────────
# 하는 일: 주파수 대역마다 (그 소리의 에너지 ÷ 섞은 소리의 에너지)를 그린다. 정답 목소리의 비율이 "이상적인 분리"의 기준선이다.
# visualisation.stft_power: 앱의 STFT 파워 계산(수업 3강 방식)을 그대로 재사용.
edges = [0, 100, 200, 400, 800, 1600, 3200, 6400, 12800, 22050]
def band_energy(x):
    hz, _t, p = visualisation.stft_power(x.mean(0), SR)
    return np.array([p[(hz >= lo) & (hz < hi)].sum() for lo, hi in zip(edges[:-1], edges[1:])])
e_mix, e_ref, e_voc = band_energy(mix), band_energy(voice), band_energy(voc)
labels = [f"{lo}-{hi}" for lo, hi in zip(edges[:-1], edges[1:])]
xx = np.arange(len(labels)); wd = 0.38
fig, ax = plt.subplots(figsize=(9, 3.6))
ax.bar(xx - wd / 2, e_ref / e_mix, wd, label="정답 목소리 / 믹스 (이상적 비율)", color=theme.SERIES[0])
ax.bar(xx + wd / 2, e_voc / e_mix, wd, label="Demucs vocals / 믹스", color=theme.SERIES[1])
ax.set_xticks(xx); ax.set_xticklabels(labels, rotation=30); ax.set_xlabel("주파수 대역 (Hz)"); ax.set_ylabel("에너지 비율")
ax.legend(); plt.tight_layout(); plt.show()
print("대역마다 비율이 1에 가깝거나 0에 가까우면 '음역대가 달라서 분리' 가설과 맞고, 중간값이 넓게 있으면 음역대가 겹친다는 뜻이다(이 합성 입력에서).")

### 이 합성 입력에서 나온 결과 [확인: 11-06 실행, 시드 고정, 이 입력에서만 해당]
- **SDR**: 섞은 소리 그대로 **−9.45 dB**, 고정 대역통과(200~4,000 Hz) **2.98 dB**, Demucs `vocals` **21.63 dB**. → 단순한 주파수 필터는 거의 개선하지 못했고, 학습된 모델은 정답에 훨씬 가깝습니다.
- **대역별 에너지 비율**: 정답 목소리의 비율(초록)은 200~1,600 Hz에서 0.8~1에 가깝고, 1,600~12,800 Hz에서는 0.2~0.35로 **0과 1 사이**입니다. 이유는 합성 타악기(잡음)가 넓은 대역에 걸쳐 있어서 **목소리와 같은 대역을 공유**하기 때문입니다. 그런데도 Demucs의 막대(파랑)가 정답 막대와 거의 겹칩니다. → "음역대가 달라서 분리된다"만으로는 설명이 안 되고, 같은 대역을 공유해도 분리된다는 증거입니다(**이 합성 입력에서만**; 실제 곡에서는 #35의 ①~③으로 확인해야 함 [미확인]).
- **주의**: 입력이 말소리 + 합성 반주라서 실제 노래의 성능으로 일반화하지 않습니다. SDR 21.6 dB는 쉬운 입력이라 높게 나온 값일 수 있습니다 [추정].

## 7. 정리 (직접 채우는 칸)

### 오늘 코드로 확인한 사실 [확인]
- (예) `get_model("htdemucs")`는 `BagOfModels`를 돌려주고 안에는 `HTDemucs` 하나가 있다. 파라미터 약 4,198만 개.
- (예) 시간 가지와 스펙트로그램 가지가 각각 4층 인코더/디코더를 갖고, 가운데 변환기 5층이 둘을 잇는다.
- (예) `nn.LSTM`은 0개, `MultiheadAttention`은 10개(셀프 6 · 교차 4).
- (직접 적기)

### 내가 이해한 것 (내 말로)
- 
- 

### 아직 이해하지 못한 것 → 강사님께 (오늘 소회의실에서)
1. 스펙트로그램 가지와 파형 가지는 **각각 어떤 정보를 학습**하나요? 두 가지를 더해서 결과를 내는 이유는요?
2. **CNN 인코더와 Transformer의 역할**은 어떻게 나뉘나요? CNN이 이미 국소 패턴을 뽑는데 왜 Attention이 또 필요한가요?
3. 인코더–디코더에서 **같은 단계끼리 더하는 건너뛰기 연결**은 수업의 Seq2Seq(010)에서 배운 것과 무엇이 같고 다른가요?
4. **교차 어텐션**의 질의와 키는 왜 서로 다른 가지에서 오나요? 토큰 수가 다른데 어떻게 계산이 되나요?
5. 토큰 수가 2,688개인데 어텐션의 점수 표가 720만 칸입니다. 모델이 **7.8초 조각으로만 학습/동작**하는 것과 관련이 있나요?
6. 학습할 때 정답은 **어떻게 주나요**(손실 함수)? 정답 보컬이 없는 상업곡에는 왜 잘 동작하나요?
7. 이전 세대 Demucs는 LSTM을 썼는데 지금은 Transformer로 바뀐 이유가 수업의 RNN → Attention 흐름과 같은 이야기인가요?
8. (도전) 어텐션 **가중치를 직접 시각화**하고 싶은데, 어떻게 꺼낼 수 있나요?

### 다음 단계
1. **#35의 ①**: 실제 시연 곡 구간 1개의 원곡·보컬·차이 스펙트로그램(결과는 `results/`에만, 커밋 금지)
2. **Whisper 탐구**(같은 형식): 입력 로그 멜(창 400·hop 160·Mel 80개) 대 우리 그림(512·128·40), 디코더의 토큰 생성, 교차 어텐션과 단어 시간
3. 앱에서 겪은 현상을 구조로 설명하기(노래에서 VAD를 켜면 가사가 지워지는 일, 반복 루프, 언어 감지가 앞 30초만 보는 점)

### 참고 위치
- 앱 코드: `separate.py`(`_load_model`, `separate_vocals`), 검토 문서 15.4(Demucs 기술 검토), 이슈 #35
- 라이브러리 소스: `demucs/htdemucs.py`, `demucs/transformer.py` (1단계 셀에서 줄 번호 확인)
- 논문: *Hybrid Transformers for Music Source Separation* (제목만 기억, 세부는 [미확인] — 읽은 뒤 이 칸에 요약)